# EP02｜電腦怎麼找到邊緣？一張 3 × 3 小格子就夠了

像素腦底加！這是影片裡的程式碼，按上方的 **執行階段 → 全部執行**（或每格按 ▶️）就能自己跑一遍。
這次只用 NumPy 和 Pillow，不用 OpenCV，每一步都看得到。

📺 影片：[youtube.com/@pixelbraintw](https://www.youtube.com/@pixelbraintw)

## 0. 準備：載入套件、下載一張貓咪照片（CC0 授權，作者 AdinaVoicu，Wikimedia Commons）

In [ ]:
import os, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

if not os.path.exists("cat.png"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/pixelbraintw/pixelbrain-code/main/ep02-edges/cat.png", "cat.png")

def show(*imgs, titles=None, size=4, top=400):
    """把一張或多張圖並排顯示（邊緣圖取絕對值，超過 top 的都算最亮）"""
    fig, axes = plt.subplots(1, len(imgs), figsize=(size * len(imgs), size))
    for i, (ax, im) in enumerate(zip(np.atleast_1d(axes), imgs)):
        if im.dtype == bool:
            ax.imshow(~im, cmap="gray", interpolation="nearest")
        elif im.ndim == 2 and (im.max() > 255 or im.min() < 0):
            ax.imshow(np.clip(np.abs(im), 0, top), cmap="gray", vmin=0, vmax=top, interpolation="nearest")
        else:
            ax.imshow(im, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        if titles: ax.set_title(titles[i])
        ax.axis("off")
    plt.show()

## 1. 讀成灰階

`convert("L")` 是轉成灰階，每一格只剩一個亮度數字，所以形狀沒有後面那個 3。

In [ ]:
gray = np.array(Image.open("cat.png").convert("L"), dtype=float)
print(gray.shape)
show(gray)

## 2. 手算一次 Sobel 九宮格

對齊的數字兩兩相乘，九個結果全部加起來。

In [ ]:
kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
edge = np.array([[10, 10, 200],
                 [10, 10, 200],
                 [10, 10, 200]])
flat = np.full((3, 3), 200)
print("放在邊緣上:", np.sum(kx * edge))
print("放在平坦處:", np.sum(kx * flat))

## 3. 把九宮格滑過整張圖

小技巧：不是一格一格移動九宮格，而是把整張圖錯開 9 次，每次乘上九宮格裡的一個數字再疊加，結果一樣但快很多。
最外圈放不滿九宮格，所以輸出是 798 × 798。

（嚴格說這是「互相關」，數學上的卷積要先把九宮格轉 180 度；對 Sobel 來說只差正負號，邊緣強度一樣。深度學習的卷積層算的也是這種不翻轉的版本。）

In [ ]:
def filter3x3(img, k):
    h, w = img.shape
    out = np.zeros((h - 2, w - 2))
    for dy in range(3):
        for dx in range(3):
            out += k[dy, dx] * img[dy:dy + h - 2, dx:dx + w - 2]
    return out

## 4. 找直的邊：gx

正的（由暗變亮）塗黃色、負的（由亮變暗）塗藍色。

In [ ]:
kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
gx = filter3x3(gray, kx)
print(gx.min(), gx.max())

pos, neg = np.clip(gx, 0, None), np.clip(-gx, 0, None)
sign = np.clip(np.stack([pos, 0.8 * pos, neg], -1) * 255 / 400, 0, 255).astype(np.uint8)
show(np.abs(gx), sign, titles=["|gx|", "gx: + yellow, - blue"], size=5)

## 5. 找橫的邊，再用畢氏定理合起來

In [ ]:
gy = filter3x3(gray, kx.T)
mag = np.hypot(gx, gy)
show(gx, gy, mag, titles=["gx", "gy", "magnitude"])

## 6. 門檻：多強才算邊緣

In [ ]:
for T in (30, 100, 300):
    print(f"門檻 {T}: {(mag > T).mean():.1%} 的像素是邊緣")

edges = mag > 100
show(mag > 30, edges, mag > 300, titles=["mag > 30", "mag > 100", "mag > 300"])

## 🧪 自己玩玩看

1. 把 `kx` 換成 `np.array([[0, 0, 0], [-1, 0, 1], [0, 0, 0]])`（只比左右兩格），跟 Sobel 比比看，哪一個雜訊比較多？
2. 試試看 `edges = mag > 50` 或 `mag > 200`，你覺得哪個門檻最好看？
3. 上傳你自己的照片（左邊資料夾圖示 → 上傳），把 `cat.png` 換成你的檔名再跑一次，看看你的照片變成著色本是什麼樣子。

有問題歡迎到影片留言區問像素腦！